In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder

df = pd.read_csv('Titanic-Dataset.csv')
print("---Dataset Summary Info ---")
df.info()
print("\n---First 5 Rows of Data---")
print(df.head())

---Dataset Summary Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

---First 5 Rows of Data---
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4       

In [36]:
# Impute 'Age' with its median (safer than mean if values are skewed)
df['Age'] = df['Age'].fillna(df['Age'].median())

# Impute 'Embarked' with its mode (most common port of embarkation)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 'Cabin' has more than 70% missing values; we can fill missing slots with 'Unknown'
df['Cabin'] = df['Cabin'].fillna('Unknown')

print("\nMissing values remaining after imputation:")
print(df.isnull().sum())



Missing values remaining after imputation:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64


In [37]:
# Initialize LabelEncoders
le_sex = LabelEncoder()
df['Sex'] = le_sex.fit_transform(df['Sex'])

le_embarked = LabelEncoder()
df['Embarked'] = le_embarked.fit_transform(df['Embarked'].astype(str))

print("\nSample of Encoded Features:")
print(df[['Sex', 'Embarked']].head())


Sample of Encoded Features:
   Sex  Embarked
0    1         2
1    0         0
2    0         2
3    0         2
4    1         2


In [38]:
# Initialize the StandardScaler
scaler = StandardScaler()

# Scale numerical columns
df[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])

print("\nSample of Standardized Numerical Data:")
print(df[['Age', 'Fare']].head())


Sample of Standardized Numerical Data:
        Age      Fare
0 -0.565736 -0.502445
1  0.663861  0.786845
2 -0.258337 -0.488854
3  0.433312  0.420730
4  0.433312 -0.486337


In [39]:
# 1. Visualize with Boxplots and save the figure
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.boxplot(df['Age'])
plt.title('Boxplot of Standardized Age')

plt.subplot(1, 2, 2)
plt.boxplot(df['Fare'])
plt.title('Boxplot of Standardized Fare')

plt.tight_layout()
plt.savefig('outliers_boxplot.png')
plt.close()

# 2. Filter and remove outliers using the IQR formula
# Outliers for Age
Q1_age = df['Age'].quantile(0.25)
Q3_age = df['Age'].quantile(0.75)
IQR_age = Q3_age - Q1_age
lower_age = Q1_age - 1.5 * IQR_age
upper_age = Q3_age + 1.5 * IQR_age

# Outliers for Fare
Q1_fare = df['Fare'].quantile(0.25)
Q3_fare = df['Fare'].quantile(0.75)
IQR_fare = Q3_fare - Q1_fare
lower_fare = Q1_fare - 1.5 * IQR_fare
upper_fare = Q3_fare + 1.5 * IQR_fare

# Keep only rows inside the acceptable upper and lower thresholds
df_cleaned = df[(df['Age'] >= lower_age) & (df['Age'] <= upper_age) &
                (df['Fare'] >= lower_fare) & (df['Fare'] <= upper_fare)]

print(f"\nOriginal rows: {df.shape[0]}")
print(f"Cleaned dataset rows after removing outliers: {df_cleaned.shape[0]}")

# Export your completed dataset
df_cleaned.to_csv('processed_titanic_dataset.csv', index=False)
print("Your completely processed dataset has been exported!")


Original rows: 891
Cleaned dataset rows after removing outliers: 721
Your completely processed dataset has been exported!


In [40]:
from sklearn.model_selection import train_test_split

# 1. Define your target (y) and features (X)
# We drop 'Survived' because it's our target, and text/ID columns that won't help prediction
X = df_cleaned.drop(columns=['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'])
y = df_cleaned['Survived']

# 2. Split the dataset into 80% Training and 20% Testing sets
# 'random_state=42' ensures that your split is reproducible every time you run it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Data successfully split!")
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Data successfully split!
Training features shape: (576, 7)
Testing features shape: (145, 7)


In [41]:
from sklearn.ensemble import RandomForestClassifier

# 1. Initialize the machine learning model
model = RandomForestClassifier(random_state=42, n_estimators=100)

# 2. Train (fit) the model using the training data
model.fit(X_train, y_train)

print("Machine learning model training complete!")

Machine learning model training complete!


In [42]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Use the trained model to make predictions on the test dataset
y_pred = model.predict(X_test)

# 2. Calculate the overall accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"--- Model Accuracy: {accuracy * 100:.2f}% ---")

# 3. Print a detailed classification report (Precision, Recall, F1-Score)
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

# 4. Print the confusion matrix
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

--- Model Accuracy: 75.86% ---

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.83      0.79      0.81        95
           1       0.64      0.70      0.67        50

    accuracy                           0.76       145
   macro avg       0.73      0.74      0.74       145
weighted avg       0.77      0.76      0.76       145

--- Confusion Matrix ---
[[75 20]
 [15 35]]


In [43]:
# Extract feature importance values from the trained model
importances = model.feature_importances_
feature_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

print("--- Feature Importance Ranking ---")
print(feature_imp)

# Plot the feature importances as a bar chart
plt.figure(figsize=(8, 4))
feature_imp.plot(kind='bar', color='skyblue')
plt.title('Which Features Mattered Most to the AI?')
plt.ylabel('Importance Score')
plt.xlabel('Features')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.close()
print("\nFeature importance chart saved as 'feature_importance.png'!")

--- Feature Importance Ranking ---
Fare        0.291711
Age         0.267383
Sex         0.234989
Pclass      0.082773
SibSp       0.045074
Parch       0.039409
Embarked    0.038660
dtype: float64

Feature importance chart saved as 'feature_importance.png'!


In [44]:
import joblib

# Save the trained Random Forest model to a file
joblib.dump(model, 'titanic_random_forest_model.pkl')

print("Model saved successfully as 'titanic_random_forest_model.pkl'!")

Model saved successfully as 'titanic_random_forest_model.pkl'!
